# Brent's method check (Python)

Checks `thole.roots.brent` against analytic roots, mpmath, SciPy, and what the Lean theorems predict, and compares it with `bisect`.
Start JupyterLab with `workshop run -- notebook`. After changing the Rust code, restart it.

In [ ]:
import math, sys
import mpmath
from scipy import optimize
import matplotlib.pyplot as plt
from thole.roots import bisect, brent, ConvergenceError, Root

XTOL, RTOL = 2e-12, 4 * sys.float_info.epsilon   # the defaults

## 1. A first call

`x` is an end of the final bracket, not its midpoint.

In [ ]:
r = brent(lambda x: x * x - 2, 0, 2)
print(r.x, r.lo, r.hi, r.iterations)
print("error      :", abs(r.x - math.sqrt(2)))
print("allowed    :", XTOL + RTOL * abs(r.x))
print("hex (compare with the Rust notebook):", r.x.hex())

## 2. Against mpmath (50 digits) and SciPy

SciPy's `brentq` has the same default tolerances. On most problems it takes the same steps, so the two agree bit for bit and evaluate `f` the same number of times. They are not the same code, though: on the root of multiplicity nine the results differ in the last digits, and both are within the tolerance.

In [ ]:
mpmath.mp.dps = 50
problems = [
    ("x^3 - x - 2",   lambda x: x * x * x - x - 2,              1.0, 2.0),
    ("cos x - x",     lambda x: math.cos(x) - x,                0.0, 1.0),
    ("x e^x - 1",     lambda x: x * math.exp(x) - 1,            0.0, 1.0),
    ("x^2 - 2",       lambda x: x * x - 2,                      0.0, 2.0),
    ("cos x",         math.cos,                                 0.0, 2.0),
    ("e^(20(x-.9))-1", lambda x: math.exp(20 * (x - 0.9)) - 1,  0.0, 1.0),
    ("(x - 0.3)^9",   lambda x: (x - 0.3) ** 9,                 0.0, 1.0),
]
exact_roots = {
    "x^3 - x - 2":    mpmath.findroot(lambda x: x**3 - x - 2, 1.5),
    "cos x - x":      mpmath.findroot(lambda x: mpmath.cos(x) - x, 0.5),
    "x e^x - 1":      mpmath.findroot(lambda x: x * mpmath.exp(x) - 1, 0.5),
    "x^2 - 2":        mpmath.sqrt(2),
    "cos x":          mpmath.pi / 2,
    "e^(20(x-.9))-1": mpmath.mpf(9) / 10,
    "(x - 0.3)^9":    mpmath.mpf(3) / 10,
}

def counted(f):
    calls = []
    def g(x):
        calls.append(x)
        return f(x)
    return g, calls

print(f"{'f':16}{'thole':>22}{'|thole - exact|':>17}{'|scipy - exact|':>17}{'evals':>7}{'scipy':>7}  same x  ok")
for name, f, a, b in problems:
    exact = exact_roots[name]
    g, calls = counted(f)
    r = brent(g, a, b, max_iter=1000)
    s, info = optimize.brentq(f, a, b, maxiter=1000, full_output=True)
    err = abs(mpmath.mpf(r.x) - exact)
    ok = err <= XTOL + RTOL * abs(r.x)
    print(f"{name:16}{r.x:22.16f}{float(err):17.3e}{float(abs(mpmath.mpf(s) - exact)):17.3e}"
          f"{len(calls):7d}{info.function_calls:7d}  {str(r.x == s):6}  {ok}")
    assert ok and len(calls) == r.iterations + 2

## 3. What the Lean theorems predict

- `brent_mem`, `brent_signChange`: the final bracket is inside `[a, b]` and `f` changes sign over it.
- `brent_error_of_done`: the error is at most `xtol + rtol * |x|`.
- `brent_done_of_le`: with `rtol = 0`, the iteration stops after at most `2 * (b - a) / xtol - 2` steps. This bound is far from sharp; it is what follows from every step being at least `xtol / 2` long.

In [ ]:
root = math.sqrt(2)
f = lambda x: x * x - 2
a, b = 0.0, 2.0
print(f"{'xtol':>8}{'x':>22}{'error':>12}{'bracket length':>16}{'steps':>7}{'bound':>15}")
for xtol in [0.5, 0.1, 1e-2, 1e-4, 1e-8, 1e-12]:
    r = brent(f, a, b, xtol=xtol, rtol=0.0)
    bound = max(math.ceil(2 * (b - a) / xtol - 2), 0)
    print(f"{xtol:8g}{r.x:22.16f}{abs(r.x - root):12.3e}{r.hi - r.lo:16.3e}{r.iterations:7d}{bound:15d}")
    assert a <= r.lo <= r.hi <= b and r.x in (r.lo, r.hi)
    assert f(r.lo) <= 0 <= f(r.hi)
    assert abs(r.x - root) <= xtol
    assert r.iterations <= bound

## 4. Convergence, next to bisection

Record every point where `f` is evaluated. Bisection gains one bit per step. Brent's method gains little at first and then converges faster than linearly.

In [ ]:
def errors(method, f, a, b, root):
    g, calls = counted(f)
    method(g, a, b)
    return [abs(x - root) for x in calls[2:]]

f = lambda x: x * math.exp(x) - 1
root = float(exact_roots["x e^x - 1"])
for method in (bisect, brent):
    e = errors(method, f, 0.0, 1.0, root)
    # An error of zero cannot be drawn on a logarithmic axis.
    plt.semilogy(range(1, len(e) + 1), [max(v, 1e-17) for v in e], "o-", label=f"{method.__name__}: {len(e)} steps")
plt.axhline(XTOL + RTOL * root, color="gray", linestyle="--", label="tolerance")
plt.xlabel("step"); plt.ylabel("|x - root|"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

## 5. Where the interpolation does not help

A jump across zero and a root of high multiplicity: Brent's method takes more steps than bisection here, but never more than about the square of the number of bisection steps. The root of multiplicity nine needs more than the default 100 steps.

In [ ]:
hard = [
    ("jump at 0.7",   lambda x: -1.0 if x < 0.7 else 1e-3, 0.0, 1.0),
    ("(x - 0.3)^9",   lambda x: (x - 0.3) ** 9,            0.0, 1.0),
    ("cbrt x + 0.1",  lambda x: math.cbrt(x) + 0.1,       -1.0, 1.0),
]
for name, f, a, b in hard:
    rb = brent(f, a, b, max_iter=1000)
    rs = bisect(f, a, b)
    print(f"{name:14} brent: {rb.iterations:4d} steps, x={rb.x:.15g}   bisect: {rs.iterations:3d} steps, x={rs.x:.15g}")

try:
    brent(lambda x: (x - 0.3) ** 9, 0.0, 1.0)
except ConvergenceError as e:
    print("default max_iter ->", e)

## 6. Failures

In [ ]:
cases = {
    "no sign change":          lambda: brent(lambda x: x * x + 1, 0, 2),
    "a > b":                   lambda: brent(lambda x: x, 1, -1),
    "infinite endpoint":       lambda: brent(lambda x: x, -math.inf, 1),
    "negative tolerance":      lambda: brent(lambda x: x, -1, 1, xtol=-1.0),
    "f returns NaN":           lambda: brent(lambda x: math.nan, 0, 1),
    "max_iter too small":      lambda: brent(math.cos, 0, 2, max_iter=3),
    "tolerance below spacing": lambda: brent(lambda x: x * x - 2, 0, 2, xtol=0.0, rtol=0.0),
    "f raises":                lambda: brent(lambda x: 1 / 0, 0, 1),
}
for name, call in cases.items():
    try:
        print(f"{name:26} -> returned {call()!r}")
    except Exception as e:
        print(f"{name:26} -> {type(e).__name__}: {e}")

## 7. Edge cases

A root at an endpoint is found without a step. A linear function is solved by the first secant step, and the bracket stays long because the value there is exactly zero. A discontinuous function: Brent's method cannot tell a pole from a root either.

In [ ]:
for name, f, a, b in [
    ("root at a",        lambda x: x,          0.0, 1.0),
    ("root at b",        lambda x: x - 1,      0.0, 1.0),
    ("a == b on a root", lambda x: x - 1,      1.0, 1.0),
    ("linear",           lambda x: 2 * x - 1,  0.0, 4.0),
    ("pole of 1/x",      lambda x: 1 / x if x != 0 else math.inf, -1.0, 2.0),
    ("three roots",      lambda x: x**3 - x,  -2.0, 3.0),
    ("huge bracket",     lambda x: x - 1,     -sys.float_info.max, sys.float_info.max),
]:
    try:
        r = brent(f, a, b)
        print(f"{name:18} x={r.x:.15g}  [{r.lo:.15g}, {r.hi:.15g}]  steps={r.iterations}")
    except Exception as e:
        print(f"{name:18} {type(e).__name__}: {e}")

## 8. Try your own

In [ ]:
r = brent(lambda x: math.sin(x) - 0.5, 0, 1)
r.x, math.pi / 6, r.iterations